In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.6 Batching and padding

Training wants many sequences at once. Contiguous windows from one long stream
need no padding; ragged inputs (different lengths) must be padded, with a mask
so the model ignores the fake pad tokens.

In [ ]:
from pocketlm import CharTokenizer, make_batch, pad_batch

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
x, y = make_batch(data, block_size=16, batch_size=4,
                  generator=torch.Generator().manual_seed(0))
print("x", tuple(x.shape), " y", tuple(y.shape), "(y is x shifted by one)")

padded, mask = pad_batch([tok.encode("hi"), tok.encode("hello")], pad_id=0)
print("padded:\n", padded)
print("mask (True = real token):\n", mask)

**Mask leakage** is the classic bug: forget the mask and the model trains on,
and attends to, padding, learning from tokens that were never there. Always
carry the mask alongside the padded batch.

In [ ]:
assert tuple(x.shape) == (4, 16)
assert padded.shape == mask.shape
assert int(mask[0].sum()) == len(tok.encode("hi"))